# 11 — Centroid-based (MEAD) + MMR (estrattivo, implementazione custom)

Prima tecnica **nativa MDS** del benchmark che affronta esplicitamente la **ridondanza tra fonti**.
Combina due idee classiche, implementate **da zero con scikit-learn** (nessuna libreria
plug-and-play):

- **Centroid-based** (sistema *MEAD*, Radev et al., 2000): si rappresenta il cluster di articoli con
  un vettore **centroide** — la media dei vettori delle sue frasi — e si assegna a ogni frase una
  **rilevanza** pari alla sua *cosine similarity* con il centroide (quanto è vicina al contenuto
  medio del cluster).
- **Maximal Marginal Relevance** (*MMR*, Carbonell & Goldstein, 1998): selezione **greedy** che a
  ogni passo aggiunge la frase che massimizza un compromesso tra rilevanza e **novità** rispetto a
  quelle già scelte:

  > score(frase) = **λ · S(frase, centroide)** − **(1 − λ) · max** *R*(frase, frasi già scelte)

  Il parametro `λ ∈ [0, 1]` bilancia salienza e diversità: `λ = 1` ignora la ridondanza (puro
  centroide), valori più bassi penalizzano le frasi simili a quelle già selezionate. Qui `λ = 0.7`.

Nel quadro della lezione del Master la tecnica è **pienamente allineata**: MMR è trattato con la
formula esplicita tra gli approcci *optimization-based* (slide 58) e il concetto di centroide tra i
*clustering-based* (slide 34). È il principale **contributo implementativo originale** del gruppo.
In letteratura, su Multi-News, MMR supera nettamente LexRank (R-1 ≈ 44,7 vs 40,3).

## Due varianti di vettorizzazione (confronto)

Il centroide e le similarità dipendono da **come si rappresentano le frasi in vettori**: qui è
l'oggetto dell'esperimento. La logica Centroid+MMR è identica; cambia **solo** il vettorizzatore:

- **`centroid_mmr`** — **TF-IDF** (`sklearn.TfidfVectorizer`): rappresentazione lessicale sparsa,
  cattura la sovrapposizione di parole. Veloce, nessun modello neurale.
- **`centroid_mmr_bert`** — **sentence embeddings BERT** (`all-MiniLM-L6-v2` di
  sentence-transformers, lo **stesso** modello di TextRank, notebook 01): rappresentazione densa e
  *semantica*, coglie la vicinanza di significato anche senza parole in comune. È la versione
  "moderna" del centroide suggerita a lezione (§3.4/§3.7). Usa la GPU se disponibile.

Il confronto misura se una rappresentazione semantica migliora le metriche MDS rispetto al puro
TF-IDF. **Segmentazione** in frasi (`psr.summarization`, come TextRank/LexRank) e **valutazione**
(ROUGE/BLEU/METEOR di pyAutoSummarizer) restano condivise, quindi i punteggi sono confrontabili.

Come i notebook 01/02 opera su due ambiti (`SCOPE`): `sample` (campione condiviso) e `full`
(intero `complete.tab` in streaming). La variante TF-IDF è rapidissima su CPU; la variante BERT è
più pesante (encoding degli embeddings) — su `full` conviene la GPU, come TextRank. Riassunti
salvati incrementalmente con **ripresa**; metriche ricalcolabili dai soli file salvati.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401  (segmentazione delle frasi + valutatore condiviso)
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

try:
    import sklearn  # noqa: F401  (TF-IDF, cosine similarity, PCA)
except ImportError:
    %pip install scikit-learn

try:
    import sentence_transformers  # noqa: F401  (embeddings BERT per la variante centroid_mmr_bert)
except ImportError:
    %pip install sentence-transformers

try:
    import matplotlib  # noqa: F401  (grafici della sezione di spiegazione)
except ImportError:
    %pip install matplotlib

In [ ]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

SCOPE       = 'sample'   # 'sample' = campione condiviso | 'full' = intero complete.tab
N_SAMPLES   = 100        # deve combaciare con il campione creato dal notebook 00
SEED        = 42
LIMIT       = None       # es. 3 per uno smoke test rapido; None = tutti
N_SENTENCES = 11         # frasi selezionate per riassunto (mediana del corpus, come 01/02)
LAMBDA      = 0.7        # peso MMR: rilevanza (centroide) vs diversita' (novita')
STOP_WORDS  = ['en']     # solo per la segmentazione con psr; l'output resta il testo originale
ST_MODEL    = 'all-MiniLM-L6-v2'  # sentence embeddings per la variante BERT (come il notebook 01)

# Le due varianti: stesso Centroid+MMR, vettorizzatore di frasi diverso
METODO_TFIDF = 'centroid_mmr'
METODO_BERT  = 'centroid_mmr_bert'
METODI       = [METODO_TFIDF, METODO_BERT]

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
DEVICE = su.rileva_device()

print(f'Ambito (SCOPE)  : {SCOPE}')
print(f'Frasi/riassunto : {N_SENTENCES}')
print(f'Lambda MMR      : {LAMBDA}')
print(f'Varianti        : {METODI}')
print(f'Device (BERT)   : {DEVICE}')

## Generazione dei riassunti

Per ogni documento (separatore `` ||||| `` sostituito con newline dal ciclo condiviso): si istanzia
`psr.summarization` e se ne leggono le frasi originali (`s.original`, in ordine di documento). Le
frasi formano un unico pool su cui si applica la pipeline **custom**:

1. **Vettorizzazione** delle frasi — **TF-IDF** (`centroid_mmr`) oppure **embeddings BERT**
   (`centroid_mmr_bert`, modello `all-MiniLM-L6-v2` caricato una sola volta). Entrambe le
   rappresentazioni sono L2-normalizzate → prodotto scalare = cosine.
2. **Centroide** = media dei vettori; **rilevanza** di ogni frase = cosine col centroide.
3. **MMR greedy**: si aggiunge iterativamente la frase che massimizza
   `λ·rilevanza − (1−λ)·max(similarita' con le frasi gia' scelte)`, fino a `N_SENTENCES` frasi.
4. Frasi selezionate **riordinate in ordine di documento** e concatenate.

Le due varianti vengono generate in sequenza in file separati
(`results/summaries/{metodo}_{scope}.tsv`), ciascuna con **ripresa** automatica. Casi limite: se le
frasi sono ≤ `N_SENTENCES` si restituiscono tutte; se il vettorizzatore fallisce (vocabolario TF-IDF
vuoto su testi minimi) si ricade sulle prime `N_SENTENCES` frasi.

In [ ]:
import numpy as np
from pyAutoSummarizer.base import psr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def segmenta(documento):
    """Frasi del cluster con la segmentazione di psr.summarization (come TextRank/LexRank).

    `s.original` e' la lista delle frasi originali in ordine di documento (split
    abbreviation-aware). Si scartano le stringhe vuote.
    """
    s = psr.summarization(documento, stop_words=STOP_WORDS)
    return [f.strip() for f in getattr(s, 'original', []) if f and f.strip()]


# --- Due vettorizzatori di frasi (unica differenza tra le varianti) ---------
def vettorizza_tfidf(frasi):
    """Matrice TF-IDF sparsa (n, vocab), righe L2-normalizzate."""
    return TfidfVectorizer(stop_words='english').fit_transform(frasi)


_bert = {}


def vettorizza_bert(frasi):
    """Embeddings Sentence-BERT densi (n, dim), L2-normalizzati. Modello caricato una volta."""
    if 'model' not in _bert:
        from sentence_transformers import SentenceTransformer
        _bert['model'] = SentenceTransformer(ST_MODEL, device=DEVICE)
    return _bert['model'].encode(frasi, normalize_embeddings=True, show_progress_bar=False)


VETTORIZZATORI = {METODO_TFIDF: vettorizza_tfidf, METODO_BERT: vettorizza_bert}


def analizza_mmr(frasi, vettorizza=vettorizza_tfidf, lam=None, k=None):
    """Esegue Centroid+MMR restituendo gli artefatti intermedi (per spiegazione/plot).

    Ritorna un dict con: matrice dei vettori `X` (TF-IDF o embeddings), `centroide`,
    `rilevanza` di ogni frase (cosine col centroide), matrice `sim` frase-frase e `scelti`
    (indici nell'ordine di selezione greedy). Con `vettorizza`/`lam`/`k` si riesegue sullo
    stesso pool con rappresentazione o parametri diversi (usato dalle figure di spiegazione).
    """
    lam = LAMBDA if lam is None else lam
    k = N_SENTENCES if k is None else k
    X = vettorizza(frasi)
    centroide = np.asarray(X.mean(axis=0)).reshape(1, -1)          # (1, dim)
    rilevanza = cosine_similarity(X, centroide).ravel()            # S(frase, centroide)
    sim = cosine_similarity(X)                                     # R(frase_i, frase_j), n x n

    scelti, candidati = [], list(range(len(frasi)))
    while len(scelti) < k and candidati:
        migliore, punteggio_migliore = None, -np.inf
        for i in candidati:
            ridondanza = max((sim[i][j] for j in scelti), default=0.0)
            punteggio = lam * rilevanza[i] - (1.0 - lam) * ridondanza
            if punteggio > punteggio_migliore:
                punteggio_migliore, migliore = punteggio, i
        scelti.append(migliore)
        candidati.remove(migliore)
    return {'X': X, 'centroide': centroide, 'rilevanza': rilevanza, 'sim': sim, 'scelti': scelti}


def make_genera(vettorizza):
    """Crea genera(documento) per un dato vettorizzatore (TF-IDF o BERT)."""
    def genera(documento):
        frasi = segmenta(documento)
        if len(frasi) <= N_SENTENCES:
            return ' '.join(frasi)
        try:
            indici = sorted(analizza_mmr(frasi, vettorizza=vettorizza)['scelti'])
        except ValueError:            # vocabolario TF-IDF vuoto su testi minimi
            indici = range(N_SENTENCES)
        return ' '.join(frasi[i] for i in indici)
    return genera


for metodo in METODI:
    out_path = P['summaries_dir'] / f'{metodo}_{SCOPE}.tsv'
    esempi = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
              else su.itera_complete_tab(P['complete_tab']))
    scrittore = su.ScrittoreRiassunti(out_path)
    print(f'== {metodo} -> {out_path.name} ==')
    su.ciclo_summarization(esempi, scrittore, make_genera(VETTORIZZATORI[metodo]),
                           limit=LIMIT, etichetta=f'{metodo} ')
    scrittore.chiudi()

## Come funziona, passo per passo

Questa sezione apre la "scatola nera" della pipeline su **un singolo documento-esempio** del
campione (parametro `RIGA_DEMO`) e con un **vettorizzatore a scelta** (`VETTORIZZAZIONE_DEMO`:
TF-IDF o BERT), per mostrare *cosa* fa Centroid+MMR e *perché*. È puramente illustrativa: non incide
sui riassunti/metriche prodotti sopra.

I quattro passi:

1. **Segmentazione** — il documento viene diviso in frasi con `psr.summarization`. È il *pool di
   candidati*.
2. **Vettorizzazione + centroide** — ogni frase diventa un vettore (TF-IDF sparso oppure embedding
   BERT denso); il **centroide** è la loro media, il "contenuto medio" del cluster (idea *MEAD*). È
   **uno solo** — non è un clustering a più centroidi.
3. **Rilevanza** — quanto ogni frase è vicina al centroide (cosine similarity): le frasi centrali
   sono le più salienti.
4. **Selezione greedy MMR** — si aggiunge una frase alla volta massimizzando
   `λ·rilevanza − (1−λ)·max(similarità con le frasi già scelte)`: la rilevanza premia le frasi
   salienti, il secondo termine penalizza quelle **ridondanti**. `λ` regola il compromesso.

Le tre figure che seguono (nello spazio del vettorizzatore scelto):

- **Scatter PCA** — le frasi proiettate in 2D, con il **centroide** (stella) al centro; colore =
  rilevanza, bordo nero = frasi scelte. Le selezionate risultano *sparse* (diversità). La PCA centra
  i dati sulla media, quindi il centroide cade vicino all'origine: rappresenta il centro del cluster.
- **Heatmap** — la similarità coseno tra tutte le frasi: i blocchi caldi fuori diagonale sono coppie
  **ridondanti**, ciò che MMR evita di scegliere due volte. Con gli embeddings BERT la similarità è
  *semantica*, quindi coglie ridondanze anche senza parole in comune.
- **Effetto di λ** — quali frasi entrano al variare di λ (1.0 = pura rilevanza, valori più bassi =
  più diversità); sotto, la **ridondanza media** del set scelto, che cala al diminuire di λ.

In [ ]:
# --- Spiegazione e figure su un documento-esempio ---------------------------
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Parametri della dimostrazione
RIGA_DEMO            = 0               # indice del documento nel campione da ispezionare
VETTORIZZAZIONE_DEMO = METODO_TFIDF    # METODO_TFIDF o METODO_BERT (spazio da visualizzare)
LAMBDA_CONFRONTO     = [1.0, 0.7, 0.5] # valori di lambda per la figura "effetto di lambda"
SALVA_FIGURE         = True            # salva anche i PNG in results/figures/centroid_mmr/
FIG_DIR              = P['metrics_dir'].parent / 'figures' / 'centroid_mmr'

# Palette coerente con il notebook 05
INK, MUTED, GRID, SURFACE, ACCENTO = '#0b0b0b', '#898781', '#e1e0d9', '#fcfcfb', '#0d9488'
plt.rcParams.update({'font.family': 'sans-serif', 'text.color': INK,
                     'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
                     'xtick.color': MUTED, 'ytick.color': MUTED})


def _densa(X):
    """Matrice densa numpy da TF-IDF sparso o da embeddings gia' densi."""
    return X.toarray() if hasattr(X, 'toarray') else np.asarray(X)


def _salva(fig, nome):
    if SALVA_FIGURE:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / nome, dpi=150, bbox_inches='tight', facecolor=SURFACE)
        print(f'  figura salvata: {FIG_DIR / nome}')


def ridondanza_media(sim, indici):
    """Similarita' coseno media tra le coppie del set selezionato (0 se <2 frasi)."""
    coppie = [sim[a][b] for i, a in enumerate(indici) for b in indici[i + 1:]]
    return float(np.mean(coppie)) if coppie else 0.0


vett = VETTORIZZATORI[VETTORIZZAZIONE_DEMO]

# 1) Documento-esempio -> frasi -> artefatti Centroid+MMR
esempio = su.carica_campione(SAMPLE_PATH)[RIGA_DEMO]
rid = esempio['row_id']
frasi = segmenta(su.prepara_documento(esempio['document']))
print(f"Documento row_id={rid} (split={esempio['split']}) — vettorizzazione: {VETTORIZZAZIONE_DEMO}")
print(f"{len(frasi)} frasi nel pool")

if len(frasi) < 3:
    print('Documento troppo corto per la visualizzazione: cambia RIGA_DEMO.')
else:
    art = analizza_mmr(frasi, vettorizza=vett)
    scelti = art['scelti']
    ordinate = sorted(scelti)
    m = np.zeros(len(frasi), dtype=bool)
    m[scelti] = True
    print(f"\nFrasi selezionate (ordine di documento), {len(ordinate)} su {len(frasi)}:")
    for i in ordinate:
        print(f"  [{i:>2}] {frasi[i][:160]}")

    Xd = _densa(art['X'])

    # --- Figura 1: proiezione PCA delle frasi + centroide -------------------
    pca = PCA(n_components=2, random_state=SEED)
    XY = pca.fit_transform(Xd)
    cxy = pca.transform(art['centroide'])
    norm = plt.Normalize(vmin=float(art['rilevanza'].min()), vmax=float(art['rilevanza'].max()))

    fig, ax = plt.subplots(figsize=(7.2, 5.4), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    ax.scatter(XY[~m, 0], XY[~m, 1], s=45, c=art['rilevanza'][~m], cmap='viridis',
               norm=norm, alpha=0.85, zorder=2)
    sc = ax.scatter(XY[m, 0], XY[m, 1], s=120, c=art['rilevanza'][m], cmap='viridis',
                    norm=norm, edgecolors=INK, linewidths=1.4,
                    label='frase scelta (MMR)', zorder=3)
    ax.scatter(cxy[0, 0], cxy[0, 1], marker='*', s=430, c=ACCENTO, edgecolors=INK,
               linewidths=1.0, label='centroide del cluster', zorder=4)
    for i in range(len(frasi)):
        ax.annotate(str(i), (XY[i, 0], XY[i, 1]), fontsize=7, color=INK,
                    ha='center', va='center', zorder=5)
    fig.colorbar(sc, ax=ax, shrink=0.85, label='rilevanza (cosine col centroide)')
    ax.set_title(f'Frasi e centroide ({VETTORIZZAZIONE_DEMO}, PCA 2D) — row {rid}',
                 color=INK, loc='left', fontsize=11)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(frameon=False, fontsize=8, loc='best')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    _salva(fig, f'scatter_pca_{VETTORIZZAZIONE_DEMO}_row{rid}.png')
    plt.show()

    # --- Figura 2: heatmap della similarita' frase-frase --------------------
    fig, ax = plt.subplots(figsize=(6.4, 5.6), facecolor=SURFACE)
    im = ax.imshow(art['sim'], cmap='magma', vmin=0, vmax=1)
    for i in scelti:
        ax.axhline(i, color=ACCENTO, linewidth=0.6, alpha=0.6)
        ax.axvline(i, color=ACCENTO, linewidth=0.6, alpha=0.6)
    fig.colorbar(im, ax=ax, shrink=0.85, label='cosine similarity')
    ax.set_title(f"Similarita' tra le frasi ({VETTORIZZAZIONE_DEMO}) — row {rid}",
                 color=INK, loc='left', fontsize=11)
    ax.set_xlabel('indice frase')
    ax.set_ylabel('indice frase')
    plt.tight_layout()
    _salva(fig, f'heatmap_sim_{VETTORIZZAZIONE_DEMO}_row{rid}.png')
    plt.show()
    print('Linee color acqua = frasi scelte; blocchi caldi fuori diagonale = coppie ridondanti.')

    # --- Figura 3: effetto di lambda sulla selezione ------------------------
    sel_per_lambda = {lam: analizza_mmr(frasi, vettorizza=vett, lam=lam)['scelti']
                      for lam in LAMBDA_CONFRONTO}
    M = np.zeros((len(LAMBDA_CONFRONTO), len(frasi)))
    for r, lam in enumerate(LAMBDA_CONFRONTO):
        M[r, sel_per_lambda[lam]] = 1.0

    fig, ax = plt.subplots(figsize=(9, 2.0 + 0.4 * len(LAMBDA_CONFRONTO)), facecolor=SURFACE)
    ax.imshow(M, cmap='Greens', vmin=0, vmax=1, aspect='auto')
    ax.set_yticks(range(len(LAMBDA_CONFRONTO)))
    ax.set_yticklabels([f'lambda={lam}' for lam in LAMBDA_CONFRONTO])
    ax.set_xticks(range(0, len(frasi), max(1, len(frasi) // 20)))
    ax.set_xlabel('indice frase')
    ax.set_title(f'Frasi selezionate al variare di lambda ({VETTORIZZAZIONE_DEMO}) — row {rid}',
                 color=INK, loc='left', fontsize=11)
    plt.tight_layout()
    _salva(fig, f'effetto_lambda_{VETTORIZZAZIONE_DEMO}_row{rid}.png')
    plt.show()

    print('\nRidondanza media del set selezionato per lambda (piu\' basso = meno ridondante):')
    for lam in LAMBDA_CONFRONTO:
        print(f'  lambda={lam}: {ridondanza_media(art["sim"], sel_per_lambda[lam]):.3f}')

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con la normalizzazione identica a tutti i metodi del
benchmark. Gli `overall` delle due varianti vengono stampati affiancati per il confronto diretto
**TF-IDF vs BERT**. Output: `results/metrics/{metodo}_{scope}_per_example.csv` e `…_aggregate.json`.

In [ ]:
import json

overall = {}
for metodo in METODI:
    out_path = P['summaries_dir'] / f'{metodo}_{SCOPE}.tsv'
    riassunti   = su.carica_riassunti(out_path)
    riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
                   else su.itera_complete_tab(P['complete_tab']))
    vettorizzazione = (f'sentence-BERT ({ST_MODEL})' if metodo == METODO_BERT
                       else 'sklearn TfidfVectorizer (stop_words=english)')
    config = {'n_sentences': N_SENTENCES, 'lambda_mmr': LAMBDA,
              'segmentazione': 'psr.summarization',
              'vettorizzazione': vettorizzazione,
              'rilevanza': 'cosine vs centroide (MEAD)',
              'selezione': 'MMR greedy',
              'libreria_valutazione': 'pyAutoSummarizer 1.2.0'}
    righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, metodo, SCOPE,
                                         P['metrics_dir'], config)
    overall[metodo] = aggregato['overall']

print('\nConfronto TF-IDF vs BERT (medie complessive):')
print(json.dumps(overall, indent=2))

## Ispezione qualitativa

Qualche coppia generato/riferimento per ciascuna variante: utile per vedere a occhio se gli
embeddings BERT selezionano frasi diverse dal TF-IDF e se il termine di diversità di MMR riduce la
ridondanza.

In [ ]:
# --- Ispezione qualitativa completa -----------------------------------------
# Mostra per intero (nessun troncamento): documento sorgente, riferimento e
# riassunto di CIASCUNA variante, per confrontare psr vs nltk a colpo d'occhio.
N_ISPEZIONE      = None   # quanti esempi: un intero (es. 5) oppure None = tutti
MOSTRA_DOCUMENTO = True   # False per nascondere gli articoli sorgente (output piu' corto)

riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
riassunti = {m: su.carica_riassunti(P['summaries_dir'] / f'{m}_{SCOPE}.tsv') for m in METODI}

mostrati = 0
for rif in riferimenti:
    rid = rif['row_id']
    if not all(rid in riassunti[m] for m in METODI):
        continue
    if N_ISPEZIONE is not None and mostrati >= N_ISPEZIONE:
        break
    print('=' * 100)
    print(f"row_id={rid} | split={rif['split']}")
    if MOSTRA_DOCUMENTO:
        articoli = [a.strip() for a in rif['document'].split(su.SEPARATORE_ARTICOLI) if a.strip()]
        print(f"\n----- DOCUMENTO ({len(articoli)} articoli) -----")
        for i, art in enumerate(articoli, 1):
            print(f"[articolo {i}]\n{art}\n")
    print(f"----- RIFERIMENTO -----\n{su.pulisci_riferimento(rif['summary'])}\n")
    for m in METODI:
        print(f"----- GENERATO [{m}] -----\n{riassunti[m][rid]}\n")
    mostrati += 1
print(f"\n({mostrati} esempi mostrati)")